In [36]:
import mysql.connector
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="",        
    database="miskatonic_university"
)

df = pd.read_sql("SELECT * FROM students LIMIT 10;", conn)
df


,id,person_id,modality,status,career_course,enrollment_status
0,s1,p2,ON_SITE,ACTIVE,Computer Science,ENROLLED
1,s2,p4,ON_SITE,ACTIVE,Data Analytics,ENROLLED
2,s3,p5,REMOTE,ACTIVE,Physics,ENROLLED
3,s4,p6,REMOTE,ACTIVE,Anthropology,ENROLLED
4,s5,p7,ON_SITE,ACTIVE,Mathematics,ENROLLED
5,s6,p8,ON_SITE,ACTIVE,Data Analytics,ENROLLED
6,s7,p9,REMOTE,ACTIVE,Computer Science,ENROLLED
7,s8,p10,REMOTE,ACTIVE,Mathematics,PENDING


In [25]:
df_age_avg = pd.read_sql("""
WITH AVG_AGE_PROFESSORS AS (
    SELECT 'Professors' AS Type, AVG(p.age) AS avg_age
    FROM professors pr
    JOIN persons p
        ON p.id = pr.person_id
),
AVG_AGE_STUDENTS AS (
    SELECT 'Students' AS Type, AVG(p.age) AS avg_age
    FROM students s
    JOIN persons p
        ON p.id = s.person_id
)
SELECT * FROM AVG_AGE_PROFESSORS
UNION ALL
SELECT * FROM AVG_AGE_STUDENTS;
""", conn)

df_age_avg

         Type  avg_age
0  Professors    49.50
1    Students    25.25


In [35]:
df_buildings = pd.read_sql("""
SELECT b.id, b.building_id, name AS building_name, l.location_id, l.location_name, b.total_floors
FROM buildings b
JOIN locations l
ON l.building_id = b.id
ORDER BY b.id;
""", conn)

df_buildings                           

,id,building_id,building_name,location_id,location_name,total_floors
0,b1,BLDG-A,Arkham Hall,LOC-101,Room 101,5
1,b1,BLDG-A,Arkham Hall,LOC-COF,Coffee Corner,5
2,b1,BLDG-A,Arkham Hall,LOC-102,Room 102,5
3,b1,BLDG-A,Arkham Hall,LOC-201,Lab Room A,5
4,b2,BLDG-B,Dunwich Science,LOC-GYM,Main Gym,3
5,b2,BLDG-B,Dunwich Science,LOC-LIB,Library,3
6,b2,BLDG-B,Dunwich Science,LOC-103,Room 103,3
7,b3,BLDG-C,Innsmouth Annex,LOC-CAF,Cafeteria,4
8,b3,BLDG-C,Innsmouth Annex,LOC-MTG,Meeting Room 1,4
9,b3,BLDG-C,Innsmouth Annex,LOC-PRK,Parking Lot A,4


In [34]:
# This shows students in remote modality from students table joining courses that aren't offered remote courses.is_remote = 0.
df_stud_remotes_onsite = pd.read_sql(
"""
SELECT p.first_name, p.last_name, s.modality, c.course_name, e.status
FROM enrollments e
JOIN students s  ON s.id = e.student_id
JOIN persons p   ON p.id = s.person_id
JOIN courses c   ON c.id = e.course_id
WHERE s.modality = 'REMOTE' AND c.is_remote = 0;
""", conn)

df_stud_remotes_onsite

,first_name,last_name,modality,course_name,status
0,Zara,Quinn,REMOTE,Applied Mathematics,PENDING


In [33]:
# df_Occupancy Rate

df_occupancy_rate = pd.read_sql(
"""
SELECT
    l.location_name,
    l.location_type,
    l.max_capacity,
    COUNT(r.id)                                  AS total_reservations,
    ROUND(COUNT(r.id) / l.max_capacity * 100, 2) AS occupancy_pct
FROM locations l
LEFT JOIN reservations r ON r.location_id = l.id AND r.status = 'CONFIRMED'
GROUP BY l.id, l.location_name, l.location_type, l.max_capacity
ORDER BY occupancy_pct DESC
""", conn)
df_occupancy_rate

,location_name,location_type,max_capacity,total_reservations,occupancy_pct
0,Meeting Room 1,COMMON_AREA,15,2,13.33
1,Main Gym,COMMON_AREA,80,2,2.50
2,Library,COMMON_AREA,150,3,2.00
3,Parking Lot A,COMMON_AREA,100,1,1.00
4,Cafeteria,COMMON_AREA,200,1,0.50
5,Room 101,CLASSROOM,30,0,0.00
6,Coffee Corner,COMMON_AREA,30,0,0.00
7,Room 102,CLASSROOM,25,0,0.00
8,Lab Room A,CLASSROOM,20,0,0.00
9,Room 103,CLASSROOM,35,0,0.00


In [47]:
# df_enrollments

df_enrollments_by_course = pd.read_sql(
"""
SELECT
    c.course_name,
    count(e.student_id) AS 'Students_Enrolled'
FROM enrollments e
LEFT JOIN courses c ON c.id = e.course_id
WHERE e.status = 'ACCEPTED'
GROUP BY c.course_name
ORDER BY Students_Enrolled DESC
""", conn)

df_enrollments_by_course

,course_name,Students_Enrolled
0,NoSQL & Modern Databases,3
1,Databases & Data Management,2
2,Introduction to Anthropology,1
3,Applied Mathematics,1
4,Physics Lab,1


In [49]:
# df_Reservations Log by Location

df_reservations_log = pd.read_sql(
    """
    SELECT 
    r.person_id,
    CONCAT(p.first_name, ' ', p.last_name) AS full_name
    l.location_name,
    r.status
    FROM reservations r
    LEFT JOIN persons p ON r.person_id = p.id
    LEFT JOIN locations l ON r.location_id = l.id
    """, conn)

df_reservations_log


ProgrammingError: 1064 (42000): You have an error in your SQL syntax; check the manual that corresponds to your MySQL server version for the right syntax to use near 'l.location_name,
    r.status
    FROM reservations r
    LEFT JOIN persons p ON' at line 4

### Charts from Dataframes

In [ ]:
# 1.- Occupancy Rate Line Chart

In [ ]:
# 2.- Popularity of courses (Enrollments)

In [ ]:
# 3.- Reservations Log by Location